# Embouchure Analysis

End-to-end exploration of the **MediaPipe-based embouchure tracking** alongside the audio intonation pipeline.

Workflow:
1. Record (or load) a paired audio + video session
2. Extract facial landmarks per frame
3. Compute embouchure consistency metrics
4. Correlate embouchure metrics with intonation stability
5. Visualize: embouchure timeline, jaw-vs-pitch overlay, correlation heatmap

Requires `opencv-python` and `mediapipe` to be installed.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Record audio + video

Set `RECORD_NOW = True` to record from your microphone and webcam. Otherwise point the paths at existing files in `data/recordings/`.

In [ ]:
RECORD_NOW = False
DURATION   = 10.0
SAVE_AS    = 'embouchure_demo'

if RECORD_NOW:
    import threading
    from audio.recorder import record_audio
    from video.embouchure_tracker import record_video
    from utils.config import SAMPLE_RATE

    video_result = []
    def _rec_video():
        video_result.append(record_video(duration=DURATION, filename=SAVE_AS + '_video'))
    t = threading.Thread(target=_rec_video, daemon=True)
    t.start()
    audio = record_audio(duration=DURATION, sample_rate=SAMPLE_RATE, filename=SAVE_AS)
    t.join()
    video_path = video_result[0]
    print('Audio + video recorded.')
else:
    import soundfile as sf
    from utils.config import SAMPLE_RATE
    audio_path = os.path.join('..', 'data', 'recordings', SAVE_AS + '.wav')
    video_path = os.path.join('..', 'data', 'recordings', SAVE_AS + '_video.mp4')
    audio, sr = sf.read(audio_path, dtype='float32')
    if audio.ndim > 1:
        audio = audio[:, 0]
    print(f'Loaded {audio_path} ({len(audio)/sr:.1f}s) and {video_path}')

## 2. Pitch + stability analysis

In [ ]:
from audio.pitch_detector import detect_pitch, get_note_name, hz_to_cents
from analysis.stability import (
    compute_stability_metrics,
    identify_unstable_regions,
    register_analysis,
)
from utils.config import SAMPLE_RATE

times, frequencies, _ = detect_pitch(audio, sr=SAMPLE_RATE)
metrics  = compute_stability_metrics(times, frequencies)
regions  = identify_unstable_regions(times, frequencies)
reg_data = register_analysis(times, frequencies)

valid = metrics['stability_score'][~np.isnan(metrics['stability_score'])]
print(f'Overall stability: {float(np.mean(valid)):.3f}  |  Unstable regions: {len(regions)}')

## 3. Extract facial landmarks

In [ ]:
from video.embouchure_tracker import extract_facial_landmarks, compute_embouchure_metrics

landmarks = extract_facial_landmarks(video_path)
emb       = compute_embouchure_metrics(landmarks)

for k, v in emb.items():
    print(f'  {k:25s}  {v}')

## 4. Embouchure timeline plots

In [ ]:
from visualization.plotter import (
    plot_embouchure_over_time,
    plot_embouchure_intonation_correlation,
    plot_correlation_heatmap,
)

plot_embouchure_over_time(landmarks, save=False)
plt.show()

In [ ]:
plot_embouchure_intonation_correlation(landmarks, times, frequencies, save=False)
plt.show()

## 5. Correlation analysis

In [ ]:
from analysis.correlation import (
    correlate_embouchure_intonation,
    correlate_by_register,
    identify_embouchure_changes,
)

corr = correlate_embouchure_intonation(landmarks, metrics, times, landmarks['times'])
for k, v in corr.items():
    print(f'  {k:35s}  {v}')

In [ ]:
plot_correlation_heatmap(corr, save=False)
plt.show()

In [ ]:
print('Per-register correlations (|r|):')
for reg in ('low', 'tenor', 'high'):
    rc = correlate_by_register(landmarks, reg_data, reg)
    ov = rc.get('overall_correlation', float('nan'))
    print(f'  {reg:6s}  {ov}')

## 6. Embouchure change events

Frames where any tracked metric jumps sharply between consecutive frames. Useful for spotting moments where you reset your embouchure mid-phrase.

In [ ]:
events = identify_embouchure_changes(landmarks, threshold=8.0)
print(f'{len(events)} change events detected.')
for t, name, mag in events[:20]:
    print(f'  t={t:5.2f}s  {name:14s}  Δ={mag}')

## 7. Interpretation

Use the printout below as a quick rubric.  
- `jaw_stability_vs_intonation` r > +0.4: keeping the jaw still helps your pitch  
- `lip_tension_vs_stability`    r > +0.4: tighter embouchure → steadier pitch (or vice-versa if negative)  
- Compare per-register |r|: do high notes need more embouchure control than low notes?

In [ ]:
consistency = emb['embouchure_consistency']
grade = 'Excellent' if consistency >= 0.85 else 'Good' if consistency >= 0.65 else 'Fair'
print(f'Embouchure consistency: {consistency:.3f}  ({grade})')
print(f'Face detected         : {emb["face_detected_pct"]:.1f}% of frames')
print(f'Overall correlation   : {corr["overall_correlation"]}')